In [21]:
import numpy as np
import scipy
import scipy.sparse as sparse
import matplotlib.pyplot as plt
import pandas as pd

# D4M (after you fix import path / editable install)
from D4M.assoc import Assoc
import D4M.util as util
from D4M.assoc import Assoc, readjson, val2col
import pandas as pd

print("Python:", __import__("sys").version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Sparse type:", type(sparse.coo_matrix(([], ([], [])), shape=(0, 0))).__name__)
print("D4M util:", util.__file__)


Python: 3.13.1
NumPy: 2.4.1
SciPy: 1.17.0
Sparse type: coo_matrix
D4M util: /Users/gcr/ingis.Wk/D4M.py/D4M/util.py


In [ ]:
import sys
print(sys.executable)
print(sys.path[:3])


/Users/gcr/ingis.Wk/D4M.py/.venv/bin/python
['/usr/local/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/usr/local/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/usr/local/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload']


NameError: name 'AA' is not defined

In [6]:
PATH = "/Users/gcr/ingis.Wk/FHIRSDS/aa-bundles"
AA1 = readjson(PATH + "/Adam631_Buckridge80_2f3fd555-23e0-adae-ccb8-438dca7e7304.json")
AA2 = readjson(PATH + "/Agatha2_Reichert620_5fd77197-1344-4b48-1907-712e228ce790.json")
AA3 = readjson(PATH + "/Akilah516_Jenkins714_946e334e-0ca7-71d8-df09-edbe34d162c0.json")
AA4 = readjson(PATH + "/Alicia629_Ariadna374_Flores439_c8fac0d1-a829-cad1-9fc9-18ae6d7e8f9d.json")

In [7]:
AA = AA1 + AA2 + AA3 + AA4

In [8]:
Assoc.size(AA)

(2634, 3879)

In [9]:
Encounter = AA[util.startswith("Encounter,"), :]

In [10]:
Encounter.get_row()

array(['Encounter/00039ac4-4cc7-d998-8f5c-905d9d2a351a',
       'Encounter/01187c15-bb56-037a-11d7-48a91e89156f',
       'Encounter/08a89c96-a8a4-9231-0548-6bec81b95404',
       'Encounter/0a822120-d8f6-ac05-45d3-75fc9884df5d',
       'Encounter/0cf1035f-417d-373e-b28c-be5772c66494',
       'Encounter/0e3af9e5-53ba-937e-3d69-eecd80d8c672',
       'Encounter/0e42a8cb-8cb1-340d-f8c9-eb97be051475',
       'Encounter/0eae3eb1-29fa-e89d-48cd-7495a8403acc',
       'Encounter/11ffef56-bbfb-2dad-d3bd-04ad7dd49164',
       'Encounter/1494a238-2ba0-79cc-bae6-6a8b5b2eb2a6',
       'Encounter/165f3aa9-01b6-3597-ed55-c29b4c11a4ff',
       'Encounter/16764e02-0ff3-7ba4-11e9-8ce45fa7e7c5',
       'Encounter/188384c4-6559-a45b-b092-92833b446d4f',
       'Encounter/1a0fbd13-3563-dd3d-5248-1d27e9fa496b',
       'Encounter/1a7cb9a4-fe2c-271c-8674-b4e9971df129',
       'Encounter/1c5fa82f-e2ed-6038-1fa9-ee7218ec38a4',
       'Encounter/1d4a2c75-2b54-e1c0-c404-f957cc005528',
       'Encounter/1d916182-7789

In [11]:
Assoc.nnz(AA)

np.int64(98851)

In [12]:
Start = AA[:, util.startswith("Encounter.period.start,")]

In [14]:
End = AA[:, util.startswith("Encounter.period.end,")]

In [28]:
found = (Start + End).find() 

# Case A: find() returns a tuple/list of 3 arrays: (rows, cols, vals)
if isinstance(found, (tuple, list)) and len(found) == 3 and not isinstance(found[0], (tuple, list)) and hasattr(found[0], "__len__"):
    rows, cols, vals = found
    df = pd.DataFrame({"row": rows, "col": cols, "val": vals})

# Case B: find() returns an (N x 3) iterable of triples
else:
    df = pd.DataFrame(list(found), columns=["row", "col", "val"])

df.head()


,row,col,val
0,Encounter/00039ac4-4cc7-d998-8f5c-905d9d2a351a,Encounter.period.start,2017-06-24T10:19:32-04:00
1,Encounter/01187c15-bb56-037a-11d7-48a91e89156f,Encounter.period.start,1985-12-05T09:20:44-05:00
2,Encounter/08a89c96-a8a4-9231-0548-6bec81b95404,Encounter.period.start,2019-09-20T17:35:25-04:00
3,Encounter/0a822120-d8f6-ac05-45d3-75fc9884df5d,Encounter.period.start,1982-12-23T09:20:44-05:00
4,Encounter/0cf1035f-417d-373e-b28c-be5772c66494,Encounter.period.start,1958-05-01T10:20:44-04:00


In [29]:
enc = df[
    df["row"].astype(str).str.startswith("Encounter/") &
    df["col"].isin(["Encounter.period.start", "Encounter.period.end"])
]

wide = enc.pivot_table(index="row", columns="col", values="val", aggfunc="first")

wide["start_dt"] = pd.to_datetime(wide["Encounter.period.start"], utc=True, errors="coerce")
wide["end_dt"]   = pd.to_datetime(wide["Encounter.period.end"],   utc=True, errors="coerce")

wide["duration_minutes"] = (wide["end_dt"] - wide["start_dt"]).dt.total_seconds() / 60

wide[["start_dt", "end_dt", "duration_minutes"]].head()


col,start_dt,end_dt,duration_minutes
row,,,
Encounter/00039ac4-4cc7-d998-8f5c-905d9d2a351a,2017-06-24 14:19:32+00:00,2017-06-24 14:34:32+00:00,15.000000
Encounter/01187c15-bb56-037a-11d7-48a91e89156f,1985-12-05 14:20:44+00:00,1985-12-05 15:16:24+00:00,55.666667
Encounter/08a89c96-a8a4-9231-0548-6bec81b95404,2019-09-20 21:35:25+00:00,2019-09-20 21:50:25+00:00,15.000000
Encounter/0a822120-d8f6-ac05-45d3-75fc9884df5d,1982-12-23 14:20:44+00:00,1982-12-23 14:35:44+00:00,15.000000
Encounter/0cf1035f-417d-373e-b28c-be5772c66494,1958-05-01 14:20:44+00:00,1958-05-01 15:16:48+00:00,56.066667


In [30]:
wide[wide["duration_minutes"] > 30]


col,Encounter.period.end,Encounter.period.start,start_dt,end_dt,duration_minutes
row,,,,,
Encounter/01187c15-bb56-037a-11d7-48a91e89156f,1985-12-05T10:16:24-05:00,1985-12-05T09:20:44-05:00,1985-12-05 14:20:44+00:00,1985-12-05 15:16:24+00:00,55.666667
Encounter/0cf1035f-417d-373e-b28c-be5772c66494,1958-05-01T11:16:48-04:00,1958-05-01T10:20:44-04:00,1958-05-01 14:20:44+00:00,1958-05-01 15:16:48+00:00,56.066667
Encounter/0e3af9e5-53ba-937e-3d69-eecd80d8c672,1978-06-01T10:51:10-04:00,1978-06-01T10:20:44-04:00,1978-06-01 14:20:44+00:00,1978-06-01 14:51:10+00:00,30.433333
Encounter/1a7cb9a4-fe2c-271c-8674-b4e9971df129,1977-03-02T18:02:48-05:00,1977-03-02T17:23:00-05:00,1977-03-02 22:23:00+00:00,1977-03-02 23:02:48+00:00,39.800000
Encounter/22b3797c-7efc-81dd-fa69-9d17f0546be6,2016-05-19T13:01:53-04:00,2016-05-19T10:03:15-04:00,2016-05-19 14:03:15+00:00,2016-05-19 17:01:53+00:00,178.633333
Encounter/268b0a59-28dc-a373-6954-2f7e595d1afb,1980-06-12T10:59:31-04:00,1980-06-12T10:20:44-04:00,1980-06-12 14:20:44+00:00,1980-06-12 14:59:31+00:00,38.783333
Encounter/288a530b-da25-06f7-5bed-ed44396d2336,1976-05-20T11:03:40-04:00,1976-05-20T10:20:44-04:00,1976-05-20 14:20:44+00:00,1976-05-20 15:03:40+00:00,42.933333
Encounter/29b2a733-3153-434e-a963-024b6ac9530f,2019-05-04T10:26:42-04:00,2019-05-04T07:19:32-04:00,2019-05-04 11:19:32+00:00,2019-05-04 14:26:42+00:00,187.166667
Encounter/2b4b1ef7-7c80-cd1d-3dd3-34b696bd8bae,2023-04-14T20:32:49-04:00,2023-04-14T17:35:25-04:00,2023-04-14 21:35:25+00:00,2023-04-15 00:32:49+00:00,177.400000
